In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [2]:
import pandas as pd

df = pd.read_csv('./dataset.csv')

In [3]:
# Step 1 -> convert text to numbers using textVocab
from tensorflow.keras.layers import TextVectorization

vectorize_layer = TextVectorization(
    max_tokens=100,
    split="character",
    output_mode = 'int',
    output_sequence_length=60
)

vectorize_layer.adapt(df.encrypt.to_list())

In [4]:
import tensorflow as tf


def map_fn(e):
    reshaped = tf.reshape(e, (-1, 1))
    return  tf.matmul(reshaped, tf.ones([1, reshaped.shape[0]]),)

def custom_layer(tensor):
    # return tensor
    return tf.map_fn(fn=map_fn, elems=tensor)

In [5]:
def input_layer(tensor):
    vectorizer_output = vectorize_layer(tensor)
    return tf.cast(vectorizer_output, "float32")

In [6]:
import tensorflow as tf
from tensorflow.keras.layers import Lambda, Conv2D, \
BatchNormalization, MaxPool2D, Reshape, Flatten, Dense, Dropout

# Create the model that uses the vectorize text layer
model = tf.keras.models.Sequential()

# Start by creating an explicit input layer. It needs to have a shape of
# (1,) (because we need to guarantee that there is exactly one string
# input per batch), and the dtype needs to be 'string'.
model.add(tf.keras.Input(shape=(1,), dtype=tf.string))

# The first layer in our model is the vectorization layer. After this
# layer, we have a tensor of shape (batch_size, max_len) containing
# vocab indices.
# model.add(vectorize_layer)
model.add(Lambda(input_layer))
model.add(tf.keras.layers.Lambda(custom_layer))
model.add(Reshape((60,60,1)))

model.add(Dropout(0.1))
model.add(Conv2D(4, (3,3), activation='relu', padding='same'))
model.add(MaxPool2D())
model.add(BatchNormalization())

model.add(Dropout(0.1))
model.add(Conv2D(2, (3,3), activation='relu', padding='same'))
model.add(MaxPool2D())
model.add(BatchNormalization())

model.add(Dropout(0.1))
model.add(Conv2D(4, (3,3), activation='relu', padding='same'))
model.add(MaxPool2D())
model.add(BatchNormalization())

model.add(Dropout(0.1))
model.add(Conv2D(8, (3,3), activation='relu', padding='same'))
model.add(MaxPool2D())
model.add(BatchNormalization())

model.add(Flatten())

model.add(Dropout(0.1))
model.add(Dense(units=8, activation='relu'))
model.add(BatchNormalization())

model.add(Dense(units=5, activation='softmax'))

In [7]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lambda (Lambda)             (None, 60)                0         
                                                                 
 lambda_1 (Lambda)           (None, 60, 60)            0         
                                                                 
 reshape (Reshape)           (None, 60, 60, 1)         0         
                                                                 
 dropout (Dropout)           (None, 60, 60, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 60, 60, 4)         40        
                                                                 
 max_pooling2d (MaxPooling2D  (None, 30, 30, 4)        0         
 )                                                               
                                                        

In [8]:
x_train = df[['encrypt']]
y_train = pd.get_dummies(df[['algorithm']])

In [9]:
from tensorflow.keras.callbacks import EarlyStopping

model.compile(
    loss="categorical_crossentropy", metrics=["accuracy"]
)
callback = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )
]

history = model.fit(
    x_train,
    y_train,
    epochs=10000,
    validation_split=0.2,
    callbacks=callback,
)

Epoch 1/10000
852/852 [==============================] - 26s 23ms/step - loss: 1.1381 - accuracy: 0.4563 - val_loss: 4.9083 - val_accuracy: 0.0000e+00
Epoch 2/10000
852/852 [==============================] - 15s 18ms/step - loss: 0.9806 - accuracy: 0.4833 - val_loss: 7.6241 - val_accuracy: 0.0000e+00
Epoch 3/10000
852/852 [==============================] - 15s 18ms/step - loss: 0.9666 - accuracy: 0.4935 - val_loss: 9.7585 - val_accuracy: 0.0000e+00
Epoch 4/10000
852/852 [==============================] - 15s 18ms/step - loss: 0.9631 - accuracy: 0.4936 - val_loss: 11.8031 - val_accuracy: 0.0000e+00
Epoch 5/10000
852/852 [==============================] - 16s 18ms/step - loss: 0.9598 - accuracy: 0.4970 - val_loss: 13.5700 - val_accuracy: 0.0000e+00
Epoch 6/10000
852/852 [==============================] - 15s 18ms/step - loss: 0.9571 - accuracy: 0.4983 - val_loss: 15.6710 - val_accuracy: 0.0000e+00
Epoch 7/10000
852/852 [==============================] - 15s 18ms/step - loss: 0.9566 - acc